In [ ]:
import numpy as np
import tensorflow as tf
import scipy.stats as si
import time

from pinn_wd import physics_informed_nn_wd
from PLOT import PLOT

## Adam 优化器

In [ ]:
def normalize(X, lb, ub):
    return 2.0 * (X - lb) / (ub - lb) - 1.0


class main:
    """Orchestrates data generation, PINN training, and plotting."""

    def __init__(self, Eq_name, n, m, n_iter=40001):
        self.Eq_name = Eq_name
        self.n, self.m = n, m

        # PDE parameters
        self.r  = 0.05
        self.q1, self.q2, self.q3 = 0.01, 0.02, 0.03
        self.sigma11, self.sigma22, self.sigma33 = 0.1, 0.2, 0.3
        self.alpha1, self.alpha2, self.alpha3 = 0.2, 0.3, 0.5
        self.T, self.K = 1.0, 17.5

        self.S1_range = [15, 20]
        self.S2_range = [15, 20]
        self.S3_range = [15, 20]
        self.t_range  = [0.01, 0.99]

        self.a11, self.a22, self.a33 = self.sigma11**2, self.sigma22**2, self.sigma33**2
        self.a12 = self.sigma11 * self.sigma22
        self.a13 = self.sigma11 * self.sigma33
        self.a23 = self.sigma22 * self.sigma33

        self.n_iter = n_iter

        # ── Sampling ──────────────────────────────────────────────────────────
        self.X         = self.sample_interior(self.n)
        self.Exact_u   = self.black_scholes_exact(self.X)

        self.Xzhongzhi, self.uzhongzhi = self.sample_terminal(self.m)
        self.X1_bianjie, self.X2_bianjie = self.sample_boundary(self.m)
        self.u1_bianjie = self.black_scholes_exact(self.X1_bianjie)
        self.u2_bianjie = self.black_scholes_exact(self.X2_bianjie)

        # ── Normalisation (fit on all points jointly) ─────────────────────────
        X_all   = np.vstack([self.X, self.Xzhongzhi, self.X1_bianjie, self.X2_bianjie])
        self.lb = np.min(X_all, axis=0)
        self.ub = np.max(X_all, axis=0)

        self.X_norm          = normalize(self.X,          self.lb, self.ub)
        self.Xzhongzhi_norm  = normalize(self.Xzhongzhi,  self.lb, self.ub)
        self.X1_norm         = normalize(self.X1_bianjie, self.lb, self.ub)
        self.X2_norm         = normalize(self.X2_bianjie, self.lb, self.ub)

        # ── Model ─────────────────────────────────────────────────────────────
        self.model = physics_informed_nn_wd(
            self.X_norm, self.Exact_u, self.lb, self.ub,
            self.Xzhongzhi_norm, self.uzhongzhi,
            self.X1_norm, self.u1_bianjie,
            self.X2_norm, self.u2_bianjie,
            w_sob=0.01, w_mean=0.10
        )

        print(f"Network layers: {self.model.Neural_Net.layers}")
        start = time.time()
        self.MSE_history, self.L2error_u = self.model.train(Eq_name, self.n_iter)
        print(f"Training finished in {time.time() - start:.1f}s")

        # ── Evaluation ────────────────────────────────────────────────────────
        self.u_pred          = self.model.predict(self.X_norm)
        self.zhongzhi_u_pred = self.model.predict(self.Xzhongzhi_norm)

        err_int  = (np.linalg.norm(self.Exact_u - self.u_pred, 2)
                    / np.linalg.norm(self.Exact_u, 2))
        err_term = (np.linalg.norm(self.uzhongzhi - self.zhongzhi_u_pred, 2)
                    / np.linalg.norm(self.uzhongzhi, 2))
        print(f"Relative L2 error (interior): {err_int:.2e}")
        print(f"Relative L2 error (terminal): {err_term:.2e}")

        # ── Scatter plots ─────────────────────────────────────────────────────
        Plot = PLOT(Eq_name, n, self.X, self.Exact_u, self.u_pred,
                    self.Xzhongzhi, self.uzhongzhi, self.zhongzhi_u_pred, self.n_iter)
        Plot.u_pred_exact_t()
        Plot.u_pred_exact_s1()
        Plot.MSE(self.n_iter, self.MSE_history)
        Plot.L2_error(self.n_iter, self.L2error_u)

        # ── Contour: S1-S2 at fixed S3=17.5, t=0.5 ───────────────────────────
        s1g = np.linspace(*self.S1_range, 40)
        s2g = np.linspace(*self.S2_range, 40)
        S1, S2 = np.meshgrid(s1g, s2g)
        X_grid = np.column_stack([S1.ravel(), S2.ravel(),
                                   np.full(1600, 17.5), np.full(1600, 0.5)])
        X_grid_norm  = normalize(X_grid, self.lb, self.ub)
        u_pred_grid  = self.model.predict(X_grid_norm).reshape(S1.shape)
        u_exact_grid = self.black_scholes_exact(X_grid).reshape(S1.shape)

        Plot.u_pred_s1_s2(S1, S2, u_pred_grid)
        Plot.u_exact_s1_s2(S1, S2, u_exact_grid)
        Plot.error_s1_s2(S1, S2, u_exact_grid - u_pred_grid)

        # ── Contour: S1-t at fixed S2=S3=17.5 ────────────────────────────────
        tg = np.linspace(*self.t_range, 40)
        S1t, Tt = np.meshgrid(s1g, tg)
        X_grid2 = np.column_stack([S1t.ravel(), np.full(1600, 17.5),
                                    np.full(1600, 17.5), Tt.ravel()])
        X_grid2_norm  = normalize(X_grid2, self.lb, self.ub)
        u_pred_grid2  = self.model.predict(X_grid2_norm).reshape(S1t.shape)
        u_exact_grid2 = self.black_scholes_exact(X_grid2).reshape(S1t.shape)

        Plot.u_pred_s1_t(S1t, Tt, u_pred_grid2)
        Plot.u_exact_s1_t(S1t, Tt, u_exact_grid2)
        Plot.error_s1_t(S1t, Tt, u_exact_grid2 - u_pred_grid2)

        self.model.analyze_cross_section()

    # ── Data generation ───────────────────────────────────────────────────────

    def sample_interior(self, n):
        return np.column_stack([
            np.random.uniform(*self.S1_range, n),
            np.random.uniform(*self.S2_range, n),
            np.random.uniform(*self.S3_range, n),
            np.random.uniform(*self.t_range,  n),
        ])

    def sample_terminal(self, m):
        X = np.column_stack([
            np.random.uniform(*self.S1_range, m),
            np.random.uniform(*self.S2_range, m),
            np.random.uniform(*self.S3_range, m),
            np.ones(m),
        ])
        u = np.maximum(
            X[:, 0:1]**self.alpha1 * X[:, 1:2]**self.alpha2 * X[:, 2:3]**self.alpha3 - self.K, 0
        )
        return X, u

    def sample_boundary(self, m):
        t = np.random.uniform(*self.t_range, m)
        X1 = np.column_stack([np.full(m, self.S1_range[0]),
                               np.full(m, self.S2_range[1]),
                               np.full(m, self.S3_range[1]), t])
        X2 = np.column_stack([np.full(m, self.S1_range[1]),
                               np.full(m, self.S2_range[0]),
                               np.full(m, self.S3_range[0]),
                               np.random.uniform(*self.t_range, m)])
        return X1, X2

    def black_scholes_exact(self, X):
        S1, S2, S3, t = X[:, 0:1], X[:, 1:2], X[:, 2:3], X[:, 3:4]
        sig2 = (self.a11*self.alpha1**2 + self.a22*self.alpha2**2 + self.a33*self.alpha3**2
                + 2*self.a12*self.alpha1*self.alpha2 + 2*self.a13*self.alpha1*self.alpha3
                + 2*self.a23*self.alpha2*self.alpha3)
        q_hat = (self.alpha1*(self.q1 + self.a11/2) + self.alpha2*(self.q2 + self.a22/2)
                 + self.alpha3*(self.q3 + self.a33/2) - sig2/2)
        tau = self.T - t
        d1 = (np.log(S1**self.alpha1 * S2**self.alpha2 * S3**self.alpha3 / self.K)
              + (self.r - q_hat + sig2/2) * tau) / np.sqrt(sig2 * tau)
        d2 = d1 - np.sqrt(sig2 * tau)
        return (np.exp(-q_hat * tau) * S1**self.alpha1 * S2**self.alpha2 * S3**self.alpha3
                * si.norm.cdf(d1) - np.exp(-self.r * tau) * self.K * si.norm.cdf(d2))


In [ ]:
np.random.seed(10)
tf.random.set_seed(5)

runner = main(Eq_name="三资产PINN", n=30000, m=3000)

## Ablation Study & Weight Sensitivity Analysis

回应审稿人意见 7：对增强损失函数的两个新项做消融实验和权重敏感性分析。

| 配置 | w_sob | w_mean | 说明 |
|------|-------|--------|------|
| B: +Sobolev | 0.01 | 0 | 只加梯度范数正则 |
| C: +Mean | 0 | 0.1 | 只加均值引导 |

In [ ]:
# ── 消融实验：辅助函数 ────────────────────────────────────────────────────────
import csv, time, matplotlib.pyplot as plt

def _ablation_make_data(runner_obj, seed):
    """用与主实验相同规模的数据，固定 seed 保证可复现。"""
    rng = np.random.RandomState(seed)
    n, m = runner_obj.n, runner_obj.m

    X_col = np.column_stack([
        rng.uniform(*runner_obj.S1_range, n),
        rng.uniform(*runner_obj.S2_range, n),
        rng.uniform(*runner_obj.S3_range, n),
        rng.uniform(*runner_obj.t_range,  n),
    ])
    u_col = runner_obj.black_scholes_exact(X_col)

    X_term = np.column_stack([
        rng.uniform(*runner_obj.S1_range, m),
        rng.uniform(*runner_obj.S2_range, m),
        rng.uniform(*runner_obj.S3_range, m),
        np.ones(m),
    ])
    u_term = np.maximum(
        X_term[:,0:1]**runner_obj.alpha1 * X_term[:,1:2]**runner_obj.alpha2
        * X_term[:,2:3]**runner_obj.alpha3 - runner_obj.K, 0
    )

    t_bc = rng.uniform(*runner_obj.t_range, m)
    X_bc1 = np.column_stack([np.full(m, runner_obj.S1_range[0]),
                               np.full(m, runner_obj.S2_range[1]),
                               np.full(m, runner_obj.S3_range[1]), t_bc])
    X_bc2 = np.column_stack([np.full(m, runner_obj.S1_range[1]),
                               np.full(m, runner_obj.S2_range[0]),
                               np.full(m, runner_obj.S3_range[0]),
                               rng.uniform(*runner_obj.t_range, m)])

    X_all = np.vstack([X_col, X_term, X_bc1, X_bc2])
    lb, ub = X_all.min(axis=0), X_all.max(axis=0)

    def norm(X): return normalize(X, lb, ub)
    return (norm(X_col), u_col,
            norm(X_term), u_term,
            norm(X_bc1), runner_obj.black_scholes_exact(X_bc1),
            norm(X_bc2), runner_obj.black_scholes_exact(X_bc2),
            lb, ub, X_col, u_col)


def ablation_run_one(runner_obj, w_sob, w_mean, n_iter=10000, seed=42):
    """训练一个 PINN，返回最终 Rel. L2 Error（interior points）。"""
    np.random.seed(seed); tf.random.set_seed(seed)

    (X_col, u_col, X_term, u_term,
     X_bc1, u_bc1, X_bc2, u_bc2,
     lb, ub, X_col_raw, u_col_raw) = _ablation_make_data(runner_obj, seed)

    model = physics_informed_nn_wd(
        X_col, u_col, lb, ub,
        X_term, u_term,
        X_bc1, u_bc1,
        X_bc2, u_bc2,
        w_sob=w_sob, w_mean=w_mean
    )
    # 只打印最后一次，避免刷屏
    model.train(Eq_name="ablation", n_iter=n_iter, print_every=n_iter)

    u_pred = model.predict(X_col)
    return float(np.linalg.norm(u_col - u_pred, 2) / np.linalg.norm(u_col, 2))

print("辅助函数已定义，可运行下方消融实验 cell。")

In [ ]:
# ── 1. 消融实验（Ablation） ──────────────────────────────────────────────────
ABLATION_N_ITER = 10000
ABLATION_SEED   = 42

ABLATION_CONFIGS = [
    ("B: +Sobolev", 0.01, 0.0),
    ("C: +Mean",    0.0,  0.1),
]

ablation_rows = []
print(f"{'Config':<22} {'w_sob':>8} {'w_mean':>8} {'L2 Error':>12}  Time")
print("-" * 60)
for name, w_sob, w_mean in ABLATION_CONFIGS:
    t0 = time.time()
    l2 = ablation_run_one(runner, w_sob, w_mean, n_iter=ABLATION_N_ITER, seed=ABLATION_SEED)
    elapsed = time.time() - t0
    ablation_rows.append((name, w_sob, w_mean, l2))
    print(f"{name:<22} {w_sob:>8.3f} {w_mean:>8.3f} {l2:>12.4e}  {elapsed:.0f}s")
print("-" * 60)

In [ ]:
# ── 2. 权重敏感性分析（Weight Sensitivity） ──────────────────────────────────
WSOB_GRID  = [0.001, 0.01, 0.1]
WMEAN_GRID = [0.01,  0.1,  1.0]

# w_sob sensitivity（固定 w_mean=0.1）
wsob_rows = []
print(f"Sensitivity: w_sob  (w_mean=0.1 fixed)")
print(f"{'w_sob':>10} {'L2 Error':>12}")
for ws in WSOB_GRID:
    l2 = ablation_run_one(runner, ws, 0.1, n_iter=ABLATION_N_ITER, seed=ABLATION_SEED)
    wsob_rows.append((ws, l2))
    print(f"{ws:>10.3f} {l2:>12.4e}")

# w_mean sensitivity（固定 w_sob=0.01）
wmean_rows = []
print(f"\nSensitivity: w_mean  (w_sob=0.01 fixed)")
print(f"{'w_mean':>10} {'L2 Error':>12}")
for wm in WMEAN_GRID:
    l2 = ablation_run_one(runner, 0.01, wm, n_iter=ABLATION_N_ITER, seed=ABLATION_SEED)
    wmean_rows.append((wm, l2))
    print(f"{wm:>10.3f} {l2:>12.4e}")

# ── 保存 CSV ──────────────────────────────────────────────────────────────────
with open("ablation_results.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["experiment", "config", "w_sob", "w_mean", "L2_error"])
    for name, ws, wm, l2 in ablation_rows:
        w.writerow(["ablation", name, ws, wm, l2])
    for ws, l2 in wsob_rows:
        w.writerow(["sensitivity_wsob", f"w_sob={ws}", ws, 0.1, l2])
    for wm, l2 in wmean_rows:
        w.writerow(["sensitivity_wmean", f"w_mean={wm}", 0.01, wm, l2])
print("\n结果已保存 → ablation_results.csv")

In [ ]:
# ── 3. 敏感性分析图（两幅子图）────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
ax.semilogx([r[0] for r in wsob_rows], [r[1] for r in wsob_rows],
            'bo-', linewidth=1.5, markersize=7)
ax.set_xlabel(r'$w_\mathrm{sob}$')
ax.set_ylabel('Relative L2 Error')
ax.set_title(r'Sensitivity: $w_\mathrm{sob}$  ($w_\mathrm{mean}=0.1$)')
ax.grid(True)

ax = axes[1]
ax.semilogx([r[0] for r in wmean_rows], [r[1] for r in wmean_rows],
            'rs-', linewidth=1.5, markersize=7)
ax.set_xlabel(r'$w_\mathrm{mean}$')
ax.set_ylabel('Relative L2 Error')
ax.set_title(r'Sensitivity: $w_\mathrm{mean}$  ($w_\mathrm{sob}=0.01$)')
ax.grid(True)

plt.tight_layout()
plt.savefig("sensitivity_analysis.png", dpi=300, bbox_inches='tight')
print("图已保存 → sensitivity_analysis.png")
plt.show()